# DOI + MinerU Markdown -> JSON -> OCR 清洗

本 notebook 只保留你现在需要的最小流程：
1. 用 DOI 从 Crossref 获取出版信息（题目、期刊、作者、出版日期等）
2. 读取 MinerU 解析出的 Markdown，拼成统一 JSON
3. 用 OCR 清洗代码清理段落与表格

说明：
- 下面每一段都写了需要改的路径与变量
- 只保留核心流程，其它与本任务无关的代码已删除


## 0. 环境与路径配置


In [5]:
from __future__ import annotations

import json
import re
import time
import hashlib
from pathlib import Path

import requests

# =========================
# 0) 路径与参数配置
# =========================
# 方式 A：如果你已经有一个 DOI 列表文件（json 或 txt），填这里
DOI_LIST_PATH = Path(r"H:\BaiduSyncdisk\code\get_article\doi_list.json")
# 方式 B：如果你只有少量 DOI，可以直接写在这里，并把 DOI_LIST_PATH 设为 None
DOI_LIST_INLINE = [
    # "10.xxxx/xxxxxx",
]

# MinerU 的输出根目录（每个 DOI 一个文件夹）
MD_ROOT = Path(r"H:\BaiduSyncdisk\code\get_article\pdf_analyse\mineru_outputs")

# 最终合并后的 JSON 输出位置
OUTPUT_JSON = Path(r"H:\BaiduSyncdisk\code\get_article\doi_markdown_full_info.json")

# Crossref 缓存目录（建议保留，避免重复请求）
CACHE_DIR = Path(r"H:\BaiduSyncdisk\code\get_article\crossref_cache")

# 如果你有邮箱，建议填写，Crossref 对有 Mailto 的请求更友好
MAILTO = ""  # 例如 "your_email@example.com"

# 请求参数
TIMEOUT = 20
MAX_RETRIES = 3
SLEEP_SECONDS = 0.2

CACHE_DIR.mkdir(parents=True, exist_ok=True)


## 1. 读取 DOI 列表


In [6]:
def load_doi_list(path: Path | None, inline_list: list[str]) -> list[str]:
    # 优先使用手写 DOI 列表
    if inline_list:
        return [d.strip() for d in inline_list if d.strip()]

    if path is None:
        raise ValueError("请设置 DOI_LIST_PATH 或 DOI_LIST_INLINE")
    if not path.exists():
        raise FileNotFoundError(f"找不到 DOI 列表文件: {path}")

    # 支持 JSON 列表 或 txt 一行一个 DOI
    if path.suffix.lower() == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(data, list):
            return [d.strip() for d in data if str(d).strip()]
        raise ValueError("JSON 文件应为 DOI 列表（list）")

    lines = path.read_text(encoding="utf-8").splitlines()
    return [line.strip() for line in lines if line.strip()]


doi_list = load_doi_list(DOI_LIST_PATH, DOI_LIST_INLINE)
print(f"DOI 数量: {len(doi_list)}")


FileNotFoundError: 找不到 DOI 列表文件: H:\BaiduSyncdisk\code\get_article\doi_list.json

## 2. Crossref 获取出版信息


In [ ]:
CROSSREF_API = "https://api.crossref.org/works/{}"


def doi_cache_path(doi: str) -> Path:
    # 用 hash 做文件名，避免 DOI 中的特殊字符
    h = hashlib.sha1(doi.encode("utf-8")).hexdigest()
    return CACHE_DIR / f"{h}.json"


def load_cache(doi: str):
    path = doi_cache_path(doi)
    if not path.exists():
        return None
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        if payload.get("doi") == doi:
            return payload.get("message")
    except Exception:
        return None
    return None


def save_cache(doi: str, msg):
    path = doi_cache_path(doi)
    payload = {"doi": doi, "message": msg}
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def extract_authors(msg: dict) -> list[str]:
    # Crossref author 结构比较多样，这里统一成 "Family, Given" 的格式
    authors = []
    for a in msg.get("author", []) or []:
        given = (a.get("given") or "").strip()
        family = (a.get("family") or "").strip()
        name = (a.get("name") or "").strip()
        if family and given:
            authors.append(f"{family}, {given}")
        elif family:
            authors.append(family)
        elif name:
            authors.append(name)
    return authors


def date_parts_to_str(parts) -> str:
    # parts 通常形如 [[YYYY, MM, DD]]
    if not parts:
        return ""
    first = parts[0]
    if not isinstance(first, list):
        return ""
    if len(first) == 1:
        return f"{first[0]}"
    if len(first) == 2:
        return f"{first[0]}-{first[1]:02d}"
    if len(first) >= 3:
        return f"{first[0]}-{first[1]:02d}-{first[2]:02d}"
    return ""


def extract_pub_date(msg: dict) -> str:
    # 优先顺序：印刷 > 在线 > issued
    for key in ("published-print", "published-online", "issued"):
        if key in msg:
            parts = msg.get(key, {}).get("date-parts")
            return date_parts_to_str(parts)
    return ""


def strip_xml_tags(text: str) -> str:
    if not text:
        return ""
    # Crossref 的 abstract 可能是 <jats:p>...</jats:p>
    text = re.sub(r"<[^>]+>", "", text)
    return text.strip()


def extract_pub_info(doi: str, msg: dict) -> dict:
    # 把 Crossref 的字段映射成我们需要的结构
    title_list = msg.get("title", []) or []
    journal_list = msg.get("container-title", []) or []
    return {
        "title": title_list[0] if title_list else "",
        "journalName": journal_list[0] if journal_list else "",
        "authors": extract_authors(msg),
        "pubDate": extract_pub_date(msg),
        "citations": msg.get("is-referenced-by-count"),
        "doi": doi,
        "abstract": strip_xml_tags(msg.get("abstract", "")),
    }


def get_crossref(doi: str, session: requests.Session, mem_cache: dict):
    # 先内存缓存，再本地缓存，最后才请求接口
    if doi in mem_cache:
        return mem_cache[doi]

    cached = load_cache(doi)
    if cached is not None:
        mem_cache[doi] = cached
        return cached

    url = CROSSREF_API.format(requests.utils.quote(doi))
    headers = {}
    if MAILTO:
        headers["User-Agent"] = f"get_article/1.0 (mailto:{MAILTO})"

    last_err = None
    for _ in range(MAX_RETRIES):
        try:
            resp = session.get(url, headers=headers, timeout=TIMEOUT)
            if resp.status_code == 200:
                msg = resp.json().get("message", {})
                save_cache(doi, msg)
                mem_cache[doi] = msg
                return msg
            last_err = f"HTTP {resp.status_code}"
        except Exception as exc:
            last_err = str(exc)
        time.sleep(SLEEP_SECONDS)

    print(f"Crossref 失败: {doi} -> {last_err}")
    return None


## 3. 读取 MinerU Markdown 并组装 JSON


In [ ]:
def doi_to_slug(doi: str) -> str:
    # DOI 里有 / 等特殊字符，统一转成安全文件名
    doi = doi.strip()
    doi = re.sub(r"[<>:\"/\\|?*\s]+", "_", doi)
    while "__" in doi:
        doi = doi.replace("__", "_")
    return doi.strip("_")


def find_markdown(doi: str, md_root: Path) -> Path | None:
    # MinerU 的输出结构可能略有不同，这里做多路径尝试
    slug = doi_to_slug(doi)
    candidates = [
        md_root / slug / "auto" / f"{slug}.md",
        md_root / slug / f"{slug}.md",
    ]
    for p in candidates:
        if p.exists():
            return p

    # 兜底：在 slug 文件夹里找任意 md
    extra_dir = md_root / slug
    if extra_dir.exists():
        extra = list(extra_dir.rglob("*.md"))
        if extra:
            return extra[0]
    return None


def is_table_line(line: str) -> bool:
    # 简单判断 Markdown 表格行（含多个 |）
    s = line.strip()
    if not s or "|" not in s:
        return False
    if re.match(r"^\|?[-: ]+\|[-|: ]+\|?$", s):
        return True
    return s.count("|") >= 2


def split_markdown(text: str) -> tuple[list[str], list[str]]:
    # 把 Markdown 粗略切成段落 + 表格块
    paragraphs = []
    tables = []
    buf = []
    table_buf = []

    def flush_para():
        if buf:
            paragraphs.append(" ".join(buf).strip())
            buf.clear()

    def flush_table():
        if table_buf:
            tables.append("\n".join(table_buf).strip())
            table_buf.clear()

    for line in text.splitlines():
        if is_table_line(line):
            flush_para()
            table_buf.append(line.rstrip())
            continue

        if line.strip() == "":
            flush_table()
            flush_para()
            continue

        flush_table()
        buf.append(line.strip())

    flush_table()
    flush_para()
    return paragraphs, tables


def build_records(dois: list[str]) -> list[dict]:
    session = requests.Session()
    mem_cache = {}
    records = []

    for idx, doi in enumerate(dois, start=1):
        msg = get_crossref(doi, session, mem_cache)
        pub_info = extract_pub_info(doi, msg or {})

        md_path = find_markdown(doi, MD_ROOT)
        if md_path is None:
            print(f"找不到 Markdown: {doi}")
            paragraphs, tables = [], []
        else:
            text = md_path.read_text(encoding="utf-8", errors="ignore")
            paragraphs, tables = split_markdown(text)

        article_info = {
            "title": pub_info.get("title", ""),
            "journalName": pub_info.get("journalName", ""),
            "authors": pub_info.get("authors", []),
            "pubDate": pub_info.get("pubDate", ""),
            "citations": pub_info.get("citations"),
            "doi": doi,
            "abstract": pub_info.get("abstract", ""),
            "paragraphs": paragraphs,
            "figureCaptions": [],
            "schemeCaptions": [],
            "tables": tables,
        }

        records.append({
            "id": idx,
            "article_information": article_info,
        })

    return records


records = build_records(doi_list)
OUTPUT_JSON.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"已保存: {OUTPUT_JSON}")


## 4. OCR 清洗（从 article_get.ipynb 提炼）


In [ ]:
# 说明：
# - 这里直接对上一步生成的 OUTPUT_JSON 做清洗
# - 如果你想批量处理多个 JSON 文件，可改成遍历文件夹

import json
import re
from pathlib import Path

IMAGE_RE = re.compile(r"!\[[^\]]*\]\([^\)]+\)")
HTML_TABLE_RE = re.compile(r"<\s*table\b", re.IGNORECASE)
HTML_TAG_RE = re.compile(r"</?[^>]+>")
REF_HEAD_RE = re.compile(
    r"^\s*#?\s*(references|references and notes|bibliography|literature cited)\b",
    re.IGNORECASE,
)
FIG_RE = re.compile(r"^\s*(fig\.|figure)\b", re.IGNORECASE)
TABLE_RE = re.compile(r"^\s*(table|tab\.)\b", re.IGNORECASE)
ACK_RE = re.compile(r"\backnowledg", re.IGNORECASE)
KEYWORDS_RE = re.compile(r"^\s*key\s*words?:", re.IGNORECASE)
SUPP_RE = re.compile(r"\b(supporting information|supplementary)\b", re.IGNORECASE)
FRONT_MATTER_RE = re.compile(
    r"\b(received|accepted|available online|published online|"
    r"corresponding author|e-?mail|fax|tel\.?|copyright|\(c\))\b",
    re.IGNORECASE,
)
AFFILIATION_RE = re.compile(
    r"\b(department of|university of|institute of|college of|faculty of|"
    r"school of|laboratory|centre|center)\b",
    re.IGNORECASE,
)

# 保留的 LaTeX 命令
LATEX_CMD_KEEP = {
    "mathrm",
    "mathbf",
    "pmb",
    "mathsf",
    "mathfrak",
    "boldsymbol",
    "text",
    "textbf",
    "textsf",
    "texttt",
    "rm",
    "bf",
    "it",
    "sf",
    "bar",
}

# 希腊字母替换表
GREEK_MAP = {
    "Alpha": "Alpha",
    "Beta": "Beta",
    "Gamma": "Gamma",
    "Delta": "Delta",
    "Epsilon": "Epsilon",
    "Zeta": "Zeta",
    "Eta": "Eta",
    "Theta": "Theta",
    "Iota": "Iota",
    "Kappa": "Kappa",
    "Lambda": "Lambda",
    "Mu": "Mu",
    "Nu": "Nu",
    "Xi": "Xi",
    "Omicron": "Omicron",
    "Pi": "Pi",
    "Rho": "Rho",
    "Sigma": "Sigma",
    "Tau": "Tau",
    "Upsilon": "Upsilon",
    "Phi": "Phi",
    "Chi": "Chi",
    "Psi": "Psi",
    "Omega": "Omega",
    "alpha": "alpha",
    "beta": "beta",
    "gamma": "gamma",
    "delta": "delta",
    "epsilon": "epsilon",
    "zeta": "zeta",
    "eta": "eta",
    "theta": "theta",
    "iota": "iota",
    "kappa": "kappa",
    "lambda": "lambda",
    "mu": "mu",
    "nu": "nu",
    "xi": "xi",
    "omicron": "omicron",
    "pi": "pi",
    "rho": "rho",
    "sigma": "sigma",
    "tau": "tau",
    "upsilon": "upsilon",
    "phi": "phi",
    "chi": "chi",
    "psi": "psi",
    "omega": "omega",
}


def is_reference_like(text: str) -> bool:
    # 判断参考文献的常见特征
    if re.search(r"\[\d+\]", text):
        return True
    if re.match(r"^\s*\d+\.", text):
        return True
    if re.search(r"\b\d{4}\b", text) and text.count(";") >= 1:
        return True
    if re.search(r"\b\d{4}\b", text) and re.search(
        r"\bJ\.|\bChem\.|\bOrg\.|\bInorg\.|\bCommun\.", text
    ):
        return True
    return False


def is_non_content(text: str) -> bool:
    # 过滤太短或几乎没有字母的行
    if len(text) < 3:
        return True
    letters = re.findall(r"[A-Za-z]", text)
    if not letters and len(text) < 10:
        return True
    return False


def collapse_spaced_letters(text: str) -> str:
    # 把被空格拆开的英文单词重新合并
    def _join(match):
        return re.sub(r"\s+", "", match.group(0))

    return re.sub(r"\b(?:[A-Za-z]\s+){2,}[A-Za-z]\b", _join, text)


def strip_latex(text: str) -> str:
    # 尽量保留含义，去掉多余的 LaTeX 命令
    text = text.replace("\xa0", " ")
    text = re.sub(r"\\begin\{[^}]+\}", "", text)
    text = re.sub(r"\\end\{[^}]+\}", "", text)
    text = re.sub(r"\\left\s*", "", text)
    text = re.sub(r"\\right\s*", "", text)
    text = re.sub(r"\$\$(.*?)\$\$", r"\1", text)
    text = re.sub(r"\$(.*?)\$", r"\1", text)

    for cmd in LATEX_CMD_KEEP:
        pattern = re.compile(rf"\\{cmd}\s*\{{([^}}]+)\}}")
        while True:
            new_text = pattern.sub(r"\1", text)
            if new_text == text:
                break
            text = new_text

    text = re.sub(
        r"\\(bf|it|rm|sf|mathsf|mathrm|mathbf|boldsymbol|pmb|textsf|texttt)\b",
        "",
        text,
    )
    text = re.sub(r"\\([A-Za-z]+)\b", lambda m: GREEK_MAP.get(m.group(1), m.group(1)), text)
    text = text.replace(r"\cdot", " ")
    text = text.replace(r"\pm", "+/-")
    text = text.replace(r"\times", "x")
    text = text.replace(r"\circ", "deg")
    text = text.replace(r"\prime", "'")
    text = text.replace(r"\%", "%")
    text = text.replace(r"\_", "_")
    text = text.replace(r"\^", "^")
    text = text.replace(r"\~", " ")
    text = text.replace(r"\&", "&")
    text = re.sub(r"\\[,;:]", " ", text)
    text = re.sub(r"\\([,.;:%])", r"\1", text)
    text = re.sub(r"\\\^circ", "deg", text)
    text = re.sub(r"\\\^", "^", text)
    text = re.sub(r"\\\*", "*", text)
    text = text.replace("^circ", "deg")
    text = re.sub(r"\bleft\s*([\(\[\{])", r"\1", text)
    text = re.sub(r"\bright\s*([\)\]\}])", r"\1", text)
    text = re.sub(r"\bbegin\s+array\b", "", text)
    text = re.sub(r"\bend\s+array\b", "", text)
    text = re.sub(r"\bbegin\s+array\s+[a-zA-Z]\b", "", text)
    text = re.sub(r"\bend\s+array\s+[a-zA-Z]\b", "", text)
    text = re.sub(
        r"\b(textrm|scriptsize|scriptstyle|displaystyle|thinspace|qquad|quad|"
        r"overline|underline|cdot|nabla|widetilde|textsf|texttt|phantom|"
        r"mathtt|mathbb|mathsf|mathfrak|mathcal|bullet|dot)\b",
        "",
        text,
    )
    text = re.sub(r"[_^]\s*\{\s*([^}]+)\s*\}", lambda m: m.group(0)[0] + m.group(1), text)
    text = text.replace("{", " ").replace("}", " ")
    text = re.sub(r"\s*_\s*", "_", text)
    text = re.sub(r"\s*\^\s*", "^", text)
    text = re.sub(r"\\\s+", " ", text)
    text = collapse_spaced_letters(text)
    text = re.sub(
        r"\b(?:[A-Za-z]\s+){1,}[A-Za-z](?=_[0-9])",
        lambda m: re.sub(r"\s+", "", m.group(0)),
        text,
    )
    text = re.sub(r"(?<=\d)\s+(?=\d)", "", text)
    text = re.sub(r"\s*~\s*", " ", text)
    text = re.sub(r"(?<=\s)\^(\d+)\b", r"\1", text)
    text = re.sub(r"\b([A-Za-z])\s*\^\s*prime\b", r"\1'", text)
    text = re.sub(r"\b([A-Za-z]+)\s*\^\s*prime\b", r"\1'", text)
    text = re.sub(r"([a-z]{3,})_([0-9])", r"\1 \2", text)
    text = re.sub(r"\bcirc\b", "deg", text)
    text = re.sub(r"\bcirc_?C\b", "deg C", text)
    text = re.sub(r"\b(hphantom|vphantom)\b", "", text)
    text = text.replace("textmu", "u")
    return text


def log_action(log_fh, record_id, action, raw, cleaned=None):
    # 可选日志：记录哪些段落被丢弃/替换
    if log_fh is None:
        return
    payload = {
        "id": record_id,
        "action": action,
        "raw": raw,
        "cleaned": cleaned,
    }
    log_fh.write(json.dumps(payload, ensure_ascii=False) + "\n")


def clean_paragraphs(paragraphs, tables, log_fh, record_id):
    # 核心清洗流程：保留正文，剔除参考文献/致谢/关键词等
    new_paras = []
    new_tables = list(tables)
    total = len(paragraphs)
    in_refs = False
    drop_tail = False

    for idx, raw in enumerate(paragraphs):
        if not isinstance(raw, str):
            continue
        text = IMAGE_RE.sub("", raw).strip()
        if text != raw:
            log_action(log_fh, record_id, "remove_image", raw, text)
        if HTML_TABLE_RE.search(text):
            new_tables.append(text)
            log_action(log_fh, record_id, "move_html_table", text)
            continue
        if HTML_TAG_RE.search(text):
            stripped = HTML_TAG_RE.sub("", text).strip()
            if stripped != text:
                log_action(log_fh, record_id, "strip_html", text, stripped)
            text = stripped
        cleaned = strip_latex(text)
        if cleaned != text:
            log_action(log_fh, record_id, "latex_simplify", text, cleaned)
        text = cleaned
        if not text:
            continue

        if REF_HEAD_RE.search(text):
            if total > 0 and idx >= int(total * 0.6):
                drop_tail = True
                break
            in_refs = True
            continue

        if in_refs:
            if is_reference_like(text):
                log_action(log_fh, record_id, "drop_reference", text)
                continue
            in_refs = False

        if ACK_RE.search(text):
            log_action(log_fh, record_id, "drop_ack", text)
            continue
        if KEYWORDS_RE.search(text):
            log_action(log_fh, record_id, "drop_keywords", text)
            continue
        if SUPP_RE.search(text):
            log_action(log_fh, record_id, "drop_supporting_info", text)
            continue
        if FRONT_MATTER_RE.search(text):
            if len(text) < 200:
                log_action(log_fh, record_id, "drop_front_matter", text)
                continue
        if AFFILIATION_RE.search(text) and len(text) < 120:
            log_action(log_fh, record_id, "drop_affiliation", text)
            continue
        if TABLE_RE.search(text):
            new_tables.append(text)
            log_action(log_fh, record_id, "move_table", text)
            continue
        if FIG_RE.search(text):
            log_action(log_fh, record_id, "drop_figure_caption", text)
            continue
        if is_non_content(text):
            log_action(log_fh, record_id, "drop_non_content", text)
            continue

        new_paras.append(text)

    if drop_tail:
        log_action(log_fh, record_id, "drop_tail_after_references", "remaining_tail")

    return new_paras, new_tables


INPUT_PATH = OUTPUT_JSON
OUTPUT_PATH = OUTPUT_JSON.with_name(OUTPUT_JSON.stem + "_clean.json")

# 如果你想记录清洗日志，把 LOG_ENABLED 改为 True
LOG_ENABLED = False
LOG_PATH = OUTPUT_JSON.with_name(OUTPUT_JSON.stem + "_clean.log")


data = json.loads(INPUT_PATH.read_text(encoding="utf-8"))
log_fh = LOG_PATH.open("w", encoding="utf-8") if LOG_ENABLED else None

try:
    for idx, rec in enumerate(data):
        if not isinstance(rec, dict):
            continue
        record_id = rec.get("id", idx)
        article = rec.get("article_information", {})
        paragraphs = article.get("paragraphs") or []
        tables = article.get("tables") or []

        new_paras, new_tables = clean_paragraphs(paragraphs, tables, log_fh, record_id)
        article["paragraphs"] = new_paras
        article["tables"] = new_tables
finally:
    if log_fh is not None:
        log_fh.close()

OUTPUT_PATH.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"清洗完成: {OUTPUT_PATH}")


In [ ]:

if __name__ == "__main__":
    # 1. Build initial records
    records = build_records(DOI_LIST_INLINE)
    
    # 2. Clean records
    for rec in records:
        article = rec.get("article_information", {})
        paragraphs = article.get("paragraphs") or []
        tables = article.get("tables") or []

        new_paras, new_tables = clean_paragraphs(paragraphs, tables)
        article["paragraphs"] = new_paras
        article["tables"] = new_tables

    # 3. Save
    OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_JSON.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Done! Saved to {OUTPUT_JSON}")